# D_S3 — Unified Permutation Test (Dense SNR)

Runs permutation tests for **all metrics** on the D experiment dataset
(single unified dataset, ~88K cases).

All cases are included (no stratified subsampling) since every (family, SNR)
combination needs a p-value for the detection curve.

Five phases, each with checkpointing:

| Phase | Metrics | Speed | Cases |
|-------|---------|-------|-------|
| 1 | Core 4 + slopes + bins + covariance | Vectorized, fast | All |
| 2 | Distance covariance | O(n²) per perm, slow | All |
| 3 | Distribution (KS, Wasserstein) | Moderate | All |
| 4 | MINE (MIC/MAS/MEV/MCN/MIC−r²) | Slow | Subset (budget) |
| 5 | LOWESS R² | Very slow | Subset (budget) |

Joint test: max-Z across all metrics → p-value → classification.

Output: `output/S3/permutation_all.parquet` + phase checkpoints

In [13]:
from __future__ import annotations

import multiprocessing as mp
import os
import time
import warnings
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.stats import rankdata, ks_2samp, wasserstein_distance, pearsonr
from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess

try:
    from minepy import MINE as MINEObj
    HAS_MINEPY = True
except ImportError:
    HAS_MINEPY = False
    print('minepy not installed — Phase 4 (MINE) will be skipped.')

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw): return it

warnings.filterwarnings('ignore')

N_WORKERS = max(1, os.cpu_count() - 2)
MP_CTX = mp.get_context('fork')

In [14]:
# ── Configuration ──
S1_DIR    = Path('output/S1')
OUT_DIR   = Path('output/S3')
CKPT_DIR  = OUT_DIR / 'checkpoints'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

N_PERM    = 200
SEED_BASE = 42_000_000

# D experiment: all cases needed for detection curve → no subsampling
RUN_MODE = 'full'

# Slow phases (4: MINE, 5: LOWESS) budget per (family, SNR) group.
# With ~88K cases, running MINE/LOWESS on all is prohibitive.
# Budget = number of replicates per (family, SNR) to include in slow phases.
# Set to 'same' to use ALL cases (very slow), or an integer (e.g. 5) to subsample.
SLOW_PHASE_REPS_PER_GROUP = 5

# Metrics where lower observed value = stronger signal
LOWER_IS_SIGNAL = {'lowess_res_sd'}

## Load Data

In [15]:
cases_df = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
pts = np.load(S1_DIR / 'scatter_points.npz')
x_all = pts['x']
y_all = pts['y']
n_cases = len(cases_df)
del pts

print(f'Loaded: {n_cases:,} cases × {x_all.shape[1]} points')
print(f'\nCategory breakdown:')
print(cases_df['category'].value_counts().to_string())

# ── Subset for slow phases (4: MINE, 5: LOWESS) ──
if SLOW_PHASE_REPS_PER_GROUP == 'same':
    subset_idx = np.arange(n_cases)
else:
    rng_sub = np.random.default_rng(99)
    chosen = []
    for _, grp in cases_df.groupby(['family_id', 'snr']):
        rows = grp.index.values
        k = min(SLOW_PHASE_REPS_PER_GROUP, len(rows))
        chosen.extend(rng_sub.choice(rows, size=k, replace=False))
    subset_idx = np.sort(np.array(chosen))

n_sub = len(subset_idx)
print(f'\nSlow-phase subset → {n_sub:,} cases for MINE/LOWESS')
print(cases_df.iloc[subset_idx]['category'].value_counts().to_string())

Loaded: 44,300 cases × 500 points

Category breakdown:
category
mean_only        44220
variance_only       60
true_null           20

Slow-phase subset → 22,115 cases for MINE/LOWESS
category
mean_only        22110
variance_only        5


## Helper Functions

In [16]:
def _generate_perms(n, n_perm, seed):
    rng = np.random.default_rng(seed)
    return np.array([rng.permutation(n) for _ in range(n_perm)])


def _double_center(a):
    D = squareform(pdist(a.reshape(-1, 1)))
    return D - D.mean(axis=0, keepdims=True) - D.mean(axis=1, keepdims=True) + D.mean()


def _precompute_bins(x, n_bins=10, min_count=5, bin_type='equal_width'):
    n = len(x)
    if n < min_count * 2:
        return None
    if bin_type == 'equal_width':
        xmin, xmax = float(x.min()), float(x.max())
        if xmax <= xmin:
            return None
        edges = np.linspace(xmin, xmax, n_bins + 1)
        edges[0] -= max(abs(xmax - xmin), 1) * 1e-10
        bins = np.searchsorted(edges, x, side='right') - 1
        bins = np.clip(bins, 0, n_bins - 1)
    else:
        order = np.argsort(x, kind='mergesort')
        bins = np.empty(n, dtype=np.intp)
        per = n / n_bins
        for b in range(n_bins):
            lo = int(round(b * per))
            hi = int(round((b + 1) * per))
            bins[order[lo:hi]] = b

    unique_bins = np.unique(bins)
    masks, counts = [], []
    for b in unique_bins:
        m = bins == b
        c = int(m.sum())
        if c >= min_count:
            masks.append(m)
            counts.append(c)
    if len(masks) < 2:
        return None
    B_ind = np.array([m.astype(np.float64) for m in masks])
    return B_ind, np.array(counts, dtype=np.float64)


def _z_and_p(obs, null, direction=1):
    med = float(np.median(null))
    iqr = float(np.percentile(null, 75) - np.percentile(null, 25))
    if iqr < 1e-12:
        iqr = float(np.std(null)) * 1.35
    if iqr < 1e-12:
        return 0.0, np.zeros_like(null), med, iqr, 1.0
    if direction == -1:
        z_obs = (med - obs) / iqr
        z_null = (med - null) / iqr
    else:
        z_obs = (obs - med) / iqr
        z_null = (null - med) / iqr
    p = float(np.sum(null >= obs if direction == 1 else null <= obs) + 1) / (len(null) + 1)
    return float(z_obs), z_null, med, iqr, p

## Phase 1: Core + Vectorizable Metrics — Parallel

|r|, |ρ|, η² (equal_width), dcor, covariance, slopes (raw/std), bin metrics, segment strength

In [17]:
def compute_phase1_case(x, y, perms):
    n = len(x)
    n_perm = len(perms)
    R = {}
    y_perms = y[perms]

    # ── |r| ──
    xc = x - x.mean(); yc = y - y.mean()
    sx = np.sqrt((xc**2).sum()); sy = np.sqrt((yc**2).sum())
    if sx > 0 and sy > 0:
        yp_centered = y_perms - y_perms.mean(1, keepdims=True)
        sy_perms = np.sqrt((yp_centered**2).sum(1))
        R['abs_pearson_r'] = (abs(float((xc*yc).sum()/(sx*sy))),
                              np.abs((xc * y_perms).sum(1) / (sx * sy_perms)))
        R['abs_covariance'] = (abs(float((xc*yc).sum()/(n-1))),
                               np.abs((xc * yp_centered).sum(1)/(n-1)))
    else:
        R['abs_pearson_r'] = (0., np.zeros(n_perm))
        R['abs_covariance'] = (0., np.zeros(n_perm))

    # ── |ρ| — vectorized rankdata ──
    xr = rankdata(x).astype(np.float64); yr = rankdata(y).astype(np.float64)
    xrc = xr - xr.mean(); yrc = yr - yr.mean()
    sxr = np.sqrt((xrc**2).sum()); syr = np.sqrt((yrc**2).sum())
    if sxr > 0 and syr > 0:
        yr_perms = rankdata(y_perms, axis=1).astype(np.float64)
        yrc_perms = yr_perms - yr_perms.mean(1, keepdims=True)
        syr_perms = np.sqrt((yrc_perms**2).sum(1))
        R['abs_spearman_rho'] = (abs(float((xrc*yrc).sum()/(sxr*syr))),
                                 np.abs((xrc * yrc_perms).sum(1) / (sxr * syr_perms)))
    else:
        R['abs_spearman_rho'] = (0., np.zeros(n_perm))

    # ── Slopes (raw + standardized) ──
    sort_idx = np.argsort(x)
    x_sorted = x[sort_idx]
    y_sorted = y[sort_idx]
    yp_sorted = y_perms[:, sort_idx]

    n1, n2 = n//3, 2*n//3
    seg_slices = {'overall': slice(None), 'early': slice(None, n1),
                  'mid': slice(n1, n2), 'late': slice(n2, None)}

    for pfx, sxs, syo, syp in [
        ('raw', x_sorted, y_sorted, yp_sorted),
        ('std', None, None, None),
    ]:
        if pfx == 'std':
            xlo, xhi = x.min(), x.max()
            xn = (x - xlo) / (xhi - xlo) if xhi > xlo else np.full(n, 0.5)
            sxs = xn[sort_idx]
            ylo_p = y_perms.min(1, keepdims=True)
            yhi_p = y_perms.max(1, keepdims=True)
            yr_p = yhi_p - ylo_p
            syp = np.where(yr_p > 0, (y_perms - ylo_p) / yr_p, 0.5)[:, sort_idx]
            ylo_o, yhi_o = y.min(), y.max()
            syo = ((y - ylo_o) / (yhi_o - ylo_o) if yhi_o > ylo_o else np.full(n, 0.5))[sort_idx]

        ep_seg_obs, ep_seg_null = [], []
        for sname, slc in seg_slices.items():
            seg_x = sxs[slc]; seg_yo = syo[slc]; seg_yp = syp[:, slc]
            if len(seg_x) < 2:
                R[f'abs_{pfx}_ep_{sname}'] = (0., np.zeros(n_perm))
                R[f'abs_{pfx}_pf_{sname}'] = (0., np.zeros(n_perm))
                continue
            dx = seg_x[-1] - seg_x[0]
            if abs(dx) > 0:
                ep_o = abs(float((seg_yo[-1] - seg_yo[0]) / dx))
                ep_n = np.abs((seg_yp[:, -1] - seg_yp[:, 0]) / dx)
            else:
                ep_o = 0.; ep_n = np.zeros(n_perm)
            R[f'abs_{pfx}_ep_{sname}'] = (ep_o, ep_n)
            seg_xc = seg_x - seg_x.mean()
            denom_pf = (seg_xc**2).sum()
            if denom_pf > 0 and len(seg_x) >= 3:
                pf_o = abs(float((seg_xc * (seg_yo - seg_yo.mean())).sum() / denom_pf))
                pf_n = np.abs((seg_xc * (seg_yp - seg_yp.mean(1, keepdims=True))).sum(1) / denom_pf)
            else:
                pf_o = 0.; pf_n = np.zeros(n_perm)
            R[f'abs_{pfx}_pf_{sname}'] = (pf_o, pf_n)
            if sname != 'overall':
                ep_seg_obs.append(ep_o); ep_seg_null.append(ep_n)
        if ep_seg_null:
            R[f'{pfx}_seg_strength'] = (float(np.mean(ep_seg_obs)), np.mean(ep_seg_null, axis=0))

    # ── Bin metrics (equal-width + equal-count) ──
    for bt, abbr in [('equal_width', 'ew'), ('equal_count', 'ec')]:
        bi = _precompute_bins(x, bin_type=bt)
        y_mean = float(y.mean())
        ss_tot = float(((y - y_mean)**2).sum())

        if bi is None or ss_tot <= 0:
            for nm in [f'{abbr}_bin_eta2', f'{abbr}_bin_amp',
                       f'{abbr}_bin_bw_mean', f'{abbr}_bin_bw_early',
                       f'{abbr}_bin_bw_mid', f'{abbr}_bin_bw_late']:
                R[nm] = (0., np.zeros(n_perm))
            continue

        B_ind, bcounts = bi
        nb = len(bcounts)
        bm_obs = (B_ind @ y) / bcounts
        bm_null = (y_perms @ B_ind.T) / bcounts

        R[f'{abbr}_bin_eta2'] = (
            float((bcounts * (bm_obs - y_mean)**2).sum() / ss_tot),
            (bcounts * (bm_null - y_mean)**2).sum(1) / ss_tot)
        R[f'{abbr}_bin_amp'] = (
            float(bm_obs.max() - bm_obs.min()),
            bm_null.max(1) - bm_null.min(1))

        masks_bool = [B_ind[b].astype(bool) for b in range(nb)]
        bw_obs_arr = np.empty(nb)
        bw_null_arr = np.empty((n_perm, nb))
        for b in range(nb):
            m = masks_bool[b]
            bw_obs_arr[b] = np.percentile(y[m], 95) - np.percentile(y[m], 5)
            yb_n = y_perms[:, m]
            bw_null_arr[:, b] = np.percentile(yb_n, 95, axis=1) - np.percentile(yb_n, 5, axis=1)

        R[f'{abbr}_bin_bw_mean'] = (float(bw_obs_arr.mean()), bw_null_arr.mean(1))
        i1b, i2b = nb // 3, 2 * nb // 3
        for nm, slc in [(f'{abbr}_bin_bw_early', slice(None, max(i1b, 1))),
                        (f'{abbr}_bin_bw_mid', slice(max(i1b, 1), max(i2b, i1b + 1))),
                        (f'{abbr}_bin_bw_late', slice(max(i2b, i1b + 1), None))]:
            o = float(bw_obs_arr[slc].mean()) if len(bw_obs_arr[slc]) else 0.
            n_ = bw_null_arr[:, slc].mean(1) if bw_null_arr[:, slc].shape[1] > 0 else np.zeros(n_perm)
            R[nm] = (o, n_)

    return R

In [18]:
def _phase1_worker(i):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)
    R = compute_phase1_case(x, y, perms)

    metrics = sorted(R.keys())
    row = {}
    z_nulls = []
    for nm in metrics:
        obs_val, null_arr = R[nm]
        d = -1 if nm in LOWER_IS_SIGNAL else 1
        z_o, z_n, med, iqr, p = _z_and_p(obs_val, null_arr.astype(np.float64), d)
        z_nulls.append(z_n)
        row[f'{nm}_obs'] = obs_val
        row[f'{nm}_null_med'] = med
        row[f'{nm}_null_iqr'] = iqr
        row[f'z_{nm}'] = z_o
        row[f'p_{nm}'] = p

    max_z_null = np.stack(z_nulls).max(axis=0).astype(np.float32)
    return i, row, max_z_null, len(metrics)


phase1_max_z_null = np.empty((n_cases, N_PERM), dtype=np.float32)
phase1_summaries = [None] * n_cases
n_metrics_phase1 = 0

t0 = time.time()
done = 0

with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=MP_CTX) as pool:
    for i, row, max_z, nm_count in tqdm(
            pool.map(_phase1_worker, range(n_cases), chunksize=64),
            total=n_cases, desc=f'Phase 1 ({N_WORKERS}w)'):
        phase1_max_z_null[i] = max_z
        phase1_summaries[i] = row
        n_metrics_phase1 = nm_count
        done += 1

        if done % 2000 == 0:
            el = time.time() - t0; rate = done / el
            print(f'  {done:>7,}/{n_cases:,}  ({rate:.0f}/s, ETA {(n_cases-done)/rate/60:.1f}min)')

elapsed = time.time() - t0
print(f'Phase 1 done: {n_cases:,} cases, {n_metrics_phase1} metrics, {elapsed/60:.1f} min')
np.savez_compressed(CKPT_DIR / 'phase1.npz', max_z_null=phase1_max_z_null)

Phase 1 (8w):   4%|▍         | 1857/44300 [04:31<1:04:16, 11.00it/s]

    2,000/44,300  (7/s, ETA 95.8min)


Phase 1 (8w):   9%|▉         | 3969/44300 [08:30<42:34, 15.79it/s]  

    4,000/44,300  (8/s, ETA 85.7min)


Phase 1 (8w):  13%|█▎        | 5825/44300 [12:34<1:24:11,  7.62it/s]

...

   42,000/44,300  (8/s, ETA 4.9min)


Phase 1 (8w):  99%|█████████▉| 43905/44300 [1:33:25<00:40,  9.64it/s]

   44,000/44,300  (8/s, ETA 0.6min)


Phase 1 (8w): 100%|██████████| 44300/44300 [1:33:47<00:00,  7.87it/s]


Phase 1 done: 44,300 cases, 33 metrics, 93.8 min


## Phase 2: Distance Covariance — Parallel

In [19]:
def _phase2_worker(i):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)

    A = _double_center(x)
    B = _double_center(y)

    dcov_obs = np.sqrt(max(float((A * B).mean()), 0))
    dcor_xx = np.sqrt(max(float((A * A).mean()), 0))
    dcor_yy = np.sqrt(max(float((B * B).mean()), 0))
    dcor_obs = dcov_obs / np.sqrt(dcor_xx * dcor_yy) if dcor_xx > 0 and dcor_yy > 0 else 0.

    dcov_null = np.empty(N_PERM)
    dcor_null = np.empty(N_PERM)
    for k in range(N_PERM):
        p = perms[k]
        Bp = B[p][:, p]
        dcv = np.sqrt(max(float((A * Bp).mean()), 0))
        dcov_null[k] = dcv
        dcr_yy_p = np.sqrt(max(float((Bp * Bp).mean()), 0))
        dcor_null[k] = dcv / np.sqrt(dcor_xx * dcr_yy_p) if dcor_xx > 0 and dcr_yy_p > 0 else 0.

    row = {}
    z_nulls = []
    for nm, obs, null in [('dcov', dcov_obs, dcov_null), ('dcor', dcor_obs, dcor_null)]:
        z_o, z_n, med, iqr, pv = _z_and_p(obs, null.astype(np.float64), 1)
        z_nulls.append(z_n)
        row[f'{nm}_obs'] = obs; row[f'{nm}_null_med'] = med
        row[f'{nm}_null_iqr'] = iqr; row[f'z_{nm}'] = z_o; row[f'p_{nm}'] = pv

    max_z_null = np.stack(z_nulls).max(0).astype(np.float32)
    return i, row, max_z_null


phase2_max_z_null = np.empty((n_cases, N_PERM), dtype=np.float32)
phase2_summaries = [None] * n_cases

t0 = time.time()
done = 0

with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=MP_CTX) as pool:
    for i, row, max_z in tqdm(pool.map(_phase2_worker, range(n_cases), chunksize=64),
                               total=n_cases, desc=f'Phase 2 ({N_WORKERS}w)'):
        phase2_max_z_null[i] = max_z
        phase2_summaries[i] = row
        done += 1

        if done % 2000 == 0:
            el = time.time() - t0; rate = done / el
            print(f'  {done:>7,}/{n_cases:,}  ({rate:.1f}/s, ETA {(n_cases-done)/rate/60:.1f}min)')

elapsed = time.time() - t0
print(f'Phase 2 done: {elapsed/60:.1f} min')
np.savez_compressed(CKPT_DIR / 'phase2.npz', max_z_null=phase2_max_z_null)

Phase 2 (8w):   4%|▍         | 1989/44300 [01:19<12:53, 54.70it/s] 

    2,000/44,300  (25.0/s, ETA 28.2min)


Phase 2 (8w):   9%|▊         | 3841/44300 [02:38<20:10, 33.44it/s]

    4,000/44,300  (25.3/s, ETA 26.5min)


Phase 2 (8w):  13%|█▎        | 5825/44300 [03:50<21:13, 30.22it/s]

...

   42,000/44,300  (28.7/s, ETA 1.3min)


Phase 2 (8w):  99%|█████████▉| 43969/44300 [25:07<00:05, 59.67it/s]

   44,000/44,300  (29.2/s, ETA 0.2min)


Phase 2 (8w): 100%|██████████| 44300/44300 [25:12<00:00, 29.28it/s]


Phase 2 done: 25.2 min


## Phase 3: Distribution Metrics (KS + Wasserstein) — Parallel

In [20]:
def _phase3_worker(i):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)
    y_perms = y[perms]

    results = {}
    for bt, abbr in [('equal_width', 'ew'), ('equal_count', 'ec')]:
        bi = _precompute_bins(x, bin_type=bt)
        if bi is None or len(bi[1]) < 3:
            for nm in [f'{abbr}_dist_ks', f'{abbr}_dist_wass']:
                results[nm] = (0., np.zeros(N_PERM))
            continue

        B_ind, bcounts = bi
        nv = len(bcounts)
        low_mask = B_ind[:max(nv // 3, 1)].max(0).astype(bool)
        high_mask = B_ind[max(2 * nv // 3, nv // 3 + 1):].max(0).astype(bool)

        y_lo_o, y_hi_o = y[low_mask], y[high_mask]
        if len(y_lo_o) < 2 or len(y_hi_o) < 2:
            for nm in [f'{abbr}_dist_ks', f'{abbr}_dist_wass']:
                results[nm] = (0., np.zeros(N_PERM))
            continue

        ks_obs = float(ks_2samp(y_lo_o, y_hi_o).statistic)
        w_obs = float(wasserstein_distance(y_lo_o, y_hi_o))
        ks_null = np.empty(N_PERM); w_null = np.empty(N_PERM)
        for k in range(N_PERM):
            yl = y_perms[k, low_mask]; yh = y_perms[k, high_mask]
            ks_null[k] = ks_2samp(yl, yh).statistic
            w_null[k] = wasserstein_distance(yl, yh)

        results[f'{abbr}_dist_ks'] = (ks_obs, ks_null)
        results[f'{abbr}_dist_wass'] = (w_obs, w_null)

    row = {}
    z_nulls = []
    for nm in ['ew_dist_ks', 'ew_dist_wass', 'ec_dist_ks', 'ec_dist_wass']:
        obs, null = results[nm]
        z_o, z_n, med, iqr, pv = _z_and_p(obs, null, 1)
        z_nulls.append(z_n)
        row[f'{nm}_obs'] = obs; row[f'{nm}_null_med'] = med
        row[f'{nm}_null_iqr'] = iqr; row[f'z_{nm}'] = z_o; row[f'p_{nm}'] = pv

    max_z_null = np.stack(z_nulls).max(0).astype(np.float32)
    return i, row, max_z_null


phase3_max_z_null = np.empty((n_cases, N_PERM), dtype=np.float32)
phase3_summaries = [None] * n_cases

t0 = time.time()
done = 0

with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=MP_CTX) as pool:
    for i, row, max_z in tqdm(pool.map(_phase3_worker, range(n_cases), chunksize=64),
                               total=n_cases, desc=f'Phase 3 ({N_WORKERS}w)'):
        phase3_max_z_null[i] = max_z
        phase3_summaries[i] = row
        done += 1

        if done % 2000 == 0:
            el = time.time() - t0; rate = done / el
            print(f'  {done:>7,}/{n_cases:,}  ({rate:.1f}/s, ETA {(n_cases-done)/rate/60:.1f}min)')

elapsed = time.time() - t0
print(f'Phase 3 done: {elapsed/60:.1f} min')
np.savez_compressed(CKPT_DIR / 'phase3.npz', max_z_null=phase3_max_z_null)

Phase 3 (8w):   4%|▍         | 1985/44300 [00:42<09:02, 77.98it/s] 

    2,000/44,300  (46.6/s, ETA 15.1min)


Phase 3 (8w):   9%|▉         | 3969/44300 [01:27<25:15, 26.61it/s] 

    4,000/44,300  (45.2/s, ETA 14.8min)


Phase 3 (8w):  13%|█▎        | 5825/44300 [02:05<20:25, 31.39it/s] 

...

   42,000/44,300  (55.5/s, ETA 0.7min)


Phase 3 (8w):  99%|█████████▉| 43969/44300 [13:08<00:05, 61.30it/s]

   44,000/44,300  (55.8/s, ETA 0.1min)


Phase 3 (8w): 100%|██████████| 44300/44300 [13:10<00:00, 56.01it/s]


Phase 3 done: 13.2 min


## Phase 4: MINE (MIC/MAS/MEV/MCN/MIC−r²) — Parallel

In [21]:
n_sub = len(subset_idx)

def _compute_mine(x, y):
    mine = MINEObj(alpha=0.6, c=15)
    mine.compute_score(x, y)
    return mine.mic(), mine.mas(), mine.mev(), mine.mcn()

def _pearson_r2_safe(x, y):
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return 0.0
    r = np.corrcoef(x, y)[0, 1]
    return float(r ** 2) if np.isfinite(r) else 0.0

def _phase4_worker(i):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)

    mic_o, mas_o, mev_o, mcn_o = _compute_mine(x, y)
    r2_o = _pearson_r2_safe(x, y)

    mic_null = np.empty(N_PERM); mas_null = np.empty(N_PERM)
    mev_null = np.empty(N_PERM); mcn_null = np.empty(N_PERM)
    mic_minus_r2_null = np.empty(N_PERM)

    for k in range(N_PERM):
        yp = y[perms[k]]
        mic_null[k], mas_null[k], mev_null[k], mcn_null[k] = _compute_mine(x, yp)
        mic_minus_r2_null[k] = mic_null[k] - _pearson_r2_safe(x, yp)

    row = {}
    z_nulls = []
    for nm, obs, null in [('mic', mic_o, mic_null), ('mas', mas_o, mas_null),
                           ('mev', mev_o, mev_null), ('mcn', mcn_o, mcn_null),
                           ('mic_minus_r2', mic_o - r2_o, mic_minus_r2_null)]:
        z_o, z_n, med, iqr, pv = _z_and_p(obs, null, 1)
        z_nulls.append(z_n)
        row[f'{nm}_obs'] = obs; row[f'{nm}_null_med'] = med
        row[f'{nm}_null_iqr'] = iqr; row[f'z_{nm}'] = z_o; row[f'p_{nm}'] = pv

    max_z_null = np.stack(z_nulls).max(0).astype(np.float32)
    return i, row, max_z_null


if HAS_MINEPY:
    phase4_max_z_null = np.full((n_cases, N_PERM), np.nan, dtype=np.float32)
    phase4_summaries = {}

    t0 = time.time()
    done = 0

    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=MP_CTX) as pool:
        for i, row, max_z in tqdm(pool.map(_phase4_worker, subset_idx, chunksize=20),
                                   total=n_sub, desc=f'Phase 4 (MINE, {N_WORKERS}w)'):
            phase4_max_z_null[i] = max_z
            phase4_summaries[i] = row
            done += 1

            if done % 500 == 0:
                el = time.time() - t0; rate = done / el
                np.savez_compressed(CKPT_DIR / 'phase4_partial.npz', max_z_null=phase4_max_z_null)
                print(f'  {done:>6,}/{n_sub:,}  ({rate:.1f}/s, ETA {(n_sub-done)/rate/3600:.1f}h)  [checkpoint saved]')

    elapsed = time.time() - t0
    print(f'Phase 4 done: {n_sub:,} cases, {N_WORKERS} workers, {elapsed/3600:.1f}h')
    np.savez_compressed(CKPT_DIR / 'phase4.npz', max_z_null=phase4_max_z_null)
else:
    phase4_max_z_null = None
    phase4_summaries = {}
    print('Phase 4 skipped (minepy not available)')

Phase 4 (MINE, 8w):   2%|▏         | 500/22115 [03:56<2:28:27,  2.43it/s]

     500/22,115  (2.1/s, ETA 2.8h)  [checkpoint saved]


Phase 4 (MINE, 8w):   5%|▍         | 1000/22115 [06:32<1:09:42,  5.05it/s]

   1,000/22,115  (2.5/s, ETA 2.3h)  [checkpoint saved]


Phase 4 (MINE, 8w):   7%|▋         | 1500/22115 [09:23<50:28,  6.81it/s]  

...

  21,500/22,115  (2.6/s, ETA 0.1h)  [checkpoint saved]


Phase 4 (MINE, 8w):  99%|█████████▉| 22000/22115 [2:18:56<00:39,  2.94it/s]

  22,000/22,115  (2.6/s, ETA 0.0h)  [checkpoint saved]


Phase 4 (MINE, 8w): 100%|██████████| 22115/22115 [2:19:15<00:00,  2.65it/s]


Phase 4 done: 22,115 cases, 8 workers, 2.3h


## Phase 5: LOWESS R² — Parallel

In [22]:
LOWESS_FRAC = 0.3
LOWESS_IT = 3

def _lowess_r2(x, y):
    if len(x) < 5:
        return 0.
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    fitted = sm_lowess(ys, xs, frac=LOWESS_FRAC, it=LOWESS_IT, return_sorted=True)
    yp = np.interp(xs, fitted[:, 0], fitted[:, 1])
    ss_res = float(((ys - yp)**2).sum())
    ss_tot = float(((ys - ys.mean())**2).sum())
    return 1. - ss_res / ss_tot if ss_tot > 0 else 0.

def _phase5_worker(i):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)

    obs = _lowess_r2(x, y)
    null_arr = np.empty(N_PERM)
    for k in range(N_PERM):
        null_arr[k] = _lowess_r2(x, y[perms[k]])

    z_o, z_n, med, iqr, pv = _z_and_p(obs, null_arr, 1)
    row = {'lowess_r2_obs': obs, 'lowess_r2_null_med': med,
           'lowess_r2_null_iqr': iqr, 'z_lowess_r2': z_o, 'p_lowess_r2': pv}
    return i, row, z_n.astype(np.float32)


phase5_max_z_null = np.full((n_cases, N_PERM), np.nan, dtype=np.float32)
phase5_summaries = {}

t0 = time.time()
done = 0

with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=MP_CTX) as pool:
    for i, row, z_null in tqdm(pool.map(_phase5_worker, subset_idx, chunksize=20),
                                total=n_sub, desc=f'Phase 5 (LOWESS, {N_WORKERS}w)'):
        phase5_max_z_null[i] = z_null
        phase5_summaries[i] = row
        done += 1

        if done % 500 == 0:
            el = time.time() - t0; rate = done / el
            np.savez_compressed(CKPT_DIR / 'phase5_partial.npz', max_z_null=phase5_max_z_null)
            print(f'  {done:>6,}/{n_sub:,}  ({rate:.1f}/s, ETA {(n_sub-done)/rate/3600:.1f}h)  [checkpoint saved]')

elapsed = time.time() - t0
print(f'Phase 5 done: {n_sub:,} cases, {N_WORKERS} workers, {elapsed/3600:.1f}h')
np.savez_compressed(CKPT_DIR / 'phase5.npz', max_z_null=phase5_max_z_null)

Phase 5 (LOWESS, 8w):   2%|▏         | 500/22115 [04:11<4:08:34,  1.45it/s]

     500/22,115  (2.0/s, ETA 3.0h)  [checkpoint saved]


Phase 5 (LOWESS, 8w):   5%|▍         | 1012/22115 [07:45<2:26:35,  2.40it/s]

   1,000/22,115  (2.1/s, ETA 2.7h)  [checkpoint saved]


Phase 5 (LOWESS, 8w):   7%|▋         | 1500/22115 [11:07<2:00:55,  2.84it/s]

...

  21,500/22,115  (2.4/s, ETA 0.1h)  [checkpoint saved]


Phase 5 (LOWESS, 8w):  99%|█████████▉| 22000/22115 [2:32:34<00:36,  3.16it/s]

  22,000/22,115  (2.4/s, ETA 0.0h)  [checkpoint saved]


Phase 5 (LOWESS, 8w): 100%|██████████| 22115/22115 [2:33:11<00:00,  2.41it/s]


Phase 5 done: 22,115 cases, 8 workers, 2.6h


## Joint Test and Classification

In [23]:
# Merge all phase summaries
all_rows = []
for i in range(n_cases):
    row = {'case_id': int(cases_df['case_id'].iloc[i])}
    row.update(phase1_summaries[i])
    row.update(phase2_summaries[i])
    row.update(phase3_summaries[i])
    if i in phase4_summaries:
        row.update(phase4_summaries[i])
    if i in phase5_summaries:
        row.update(phase5_summaries[i])
    all_rows.append(row)

# Joint T from max-Z across all phases
T_null_joint = np.maximum(phase1_max_z_null, phase2_max_z_null)
T_null_joint = np.maximum(T_null_joint, phase3_max_z_null)
if phase4_max_z_null is not None:
    p4 = phase4_max_z_null.copy()
    p4[np.isnan(p4)] = -np.inf
    T_null_joint = np.maximum(T_null_joint, p4)
p5 = phase5_max_z_null.copy()
p5[np.isnan(p5)] = -np.inf
T_null_joint = np.maximum(T_null_joint, p5)

# Collect all z_ columns for T_obs
z_cols = [c for c in all_rows[0] if c.startswith('z_')]
for i, row in enumerate(all_rows):
    T_obs = max((row.get(c, -np.inf) for c in z_cols), default=0.)
    p_val = float(np.sum(T_null_joint[i] >= T_obs) + 1) / (N_PERM + 1)
    row['T_joint'] = T_obs
    row['p_value'] = p_val

print(f'Assembly done: {len(z_cols)} z-score metrics in joint test')

Assembly done: 39 z-score metrics in joint test


In [24]:
df = pd.DataFrame(all_rows)
df['classification'] = 'uncertain'
df.loc[df['p_value'] <= 0.05, 'classification'] = 'detectable'
df.loc[df['p_value'] >= 0.10, 'classification'] = 'not_detectable'

out_path = OUT_DIR / 'permutation_all.parquet'
df.to_parquet(out_path, index=False)
print(f'Saved {out_path}  ({len(df):,} rows × {len(df.columns)} cols)')
print()
print(df['classification'].value_counts().to_string())
print()
print(df.head(3))

Saved output/S3/permutation_all.parquet  (44,300 rows × 229 cols)

classification
detectable        35685
not_detectable     7630
uncertain           985

   case_id  abs_covariance_obs  abs_covariance_null_med  \
0        1            0.022590                 0.076268   
1        2            0.096567                 0.084181   
2        3            0.059745                 0.075087   

   abs_covariance_null_iqr  z_abs_covariance  p_abs_covariance  \
0                 0.094136         -0.570214          0.825871   
1                 0.106397          0.116408          0.442786   
2                 0.084983         -0.180522          0.656716   

   abs_pearson_r_obs  abs_pearson_r_null_med  abs_pearson_r_null_iqr  \
0           0.008855                0.029895                0.036899   
1           0.037512                0.032701                0.041331   
2           0.023395                0.029403                0.033278   

   z_abs_pearson_r  ...  mic_minus_r2_null_med  mic_mi